<a href="https://colab.research.google.com/github/cezmi9104-sys/Open-sourced-off-the-shelf-ion-exchange-membrane/blob/main/gemma_%C3%A7eviri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# HÜCRE 1 — Kurulum ve Gemma3 12B Modeli Yükleme
# ============================================================

!pip install -q transformers accelerate bitsandbytes pdfplumber huggingface_hub

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata  # ← Colab Secrets'tan okur

# --- Token'ı Colab Secrets'tan al ---
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Hugging Face girişi başarılı!")
except Exception as e:
    print(f"❌ Token hatası: {e}")
    print("Sol menüden 🔑 Secrets bölümüne HF_TOKEN ekleyin!")
    raise

# --- Model ayarları ---
MODEL_ID = "google/gemma-3-4b-it"

print("⏳ Model yükleniyor...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print("✅ Model hazır!")
print(f"🖥️  GPU: {next(model.parameters()).device}")


# --- Çeviri fonksiyonları (aynı kalıyor) ---
def translate_chunk(text: str, chunk_index: int = 0, total_chunks: int = 1) -> str:
    prompt = f"""Aşağıdaki metni Türkçeye çevir. Sadece çeviriyi yaz, açıklama ekleme.

METİN:
{text}

TÜRKÇE ÇEVİRİ:"""

    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    result = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    print(f"  ✔ Parça {chunk_index + 1}/{total_chunks} çevrildi.")
    return result


def split_text(text: str, max_chars: int = 2500) -> list:
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= max_chars:
            current += ("\n" if current else "") + para
        else:
            if current:
                chunks.append(current)
            if len(para) > max_chars:
                for i in range(0, len(para), max_chars):
                    chunks.append(para[i:i + max_chars])
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks

print("\n📌 Artık Hücre 2'yi çalıştırabilirsiniz!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.8 MB/s eta 0:00:00
✅ Hugging Face girişi başarılı!
⏳ Model yükleniyor...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✅ Model hazır!
🖥️  GPU: cuda:0

📌 Artık Hücre 2'yi çalıştırabilirsiniz!


In [5]:
# ============================================================
# HÜCRE 2 — PDF Yükle ve Türkçeye Çevir
# Her yeni PDF için bu hücreyi çalıştırın, model yeniden yüklenmez!
# ============================================================

import pdfplumber
import os
from google.colab import files

# --- PDF dosyasını seç ---
print("📂 Lütfen bir PDF dosyası seçin...")
uploaded = files.upload()

if not uploaded:
    print("❌ Dosya seçilmedi.")
else:
    pdf_filename = list(uploaded.keys())[0]
    print(f"\n📄 Yüklenen dosya: {pdf_filename}")

    # --- PDF'den metin çıkar ---
    print("📖 PDF okunuyor...")
    full_text = ""
    page_count = 0

    with pdfplumber.open(pdf_filename) as pdf:
        page_count = len(pdf.pages)
        print(f"   Toplam sayfa: {page_count}")

        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text()
            if page_text:
                full_text += f"\n\n--- Sayfa {i + 1} ---\n{page_text}"

    if not full_text.strip():
        print("⚠️  PDF'den metin çıkarılamadı. Taranmış (görüntü tabanlı) bir PDF olabilir.")
    else:
        print(f"   Toplam karakter: {len(full_text):,}")

        # --- Metni parçalara böl ---
        chunks = split_text(full_text, max_chars=2500)
        print(f"\n🔀 Metin {len(chunks)} parçaya bölündü.")
        print("🌐 Çeviri başlıyor...\n")

        # --- Her parçayı çevir ---
        translated_parts = []
        for i, chunk in enumerate(chunks):
            translated = translate_chunk(chunk, chunk_index=i, total_chunks=len(chunks))
            translated_parts.append(translated)

        full_translation = "\n\n".join(translated_parts)

        # --- Sonucu kaydet ---
        output_filename = os.path.splitext(pdf_filename)[0] + "_turkce.txt"
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(f"KAYNAK DOSYA: {pdf_filename}\n")
            f.write(f"SAYFA SAYISI: {page_count}\n")
            f.write("=" * 60 + "\n\n")
            f.write(full_translation)

        print(f"\n✅ Çeviri tamamlandı!")
        print(f"💾 Kaydedilen dosya: {output_filename}")

        # --- Dosyayı indir ---
        files.download(output_filename)
        print("⬇️  İndirme başlatıldı.")

        # --- Önizleme (ilk 500 karakter) ---
        print("\n" + "=" * 50)
        print("📋 ÇEVİRİ ÖNİZLEME (ilk 500 karakter):")
        print("=" * 50)
        print(full_translation[:500] + "...")

📂 Lütfen bir PDF dosyası seçin...


Saving CN115784196B.pdf to CN115784196B.pdf

📄 Yüklenen dosya: CN115784196B.pdf
📖 PDF okunuyor...
   Toplam sayfa: 11
   Toplam karakter: 11,196

🔀 Metin 5 parçaya bölündü.
🌐 Çeviri başlıyor...

  ✔ Parça 1/5 çevrildi.
  ✔ Parça 2/5 çevrildi.
  ✔ Parça 3/5 çevrildi.
  ✔ Parça 4/5 çevrildi.
  ✔ Parça 5/5 çevrildi.

✅ Çeviri tamamlandı!
💾 Kaydedilen dosya: CN115784196B_turkce.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  İndirme başlatıldı.

📋 ÇEVİRİ ÖNİZLEME (ilk 500 karakter):
(19) 国家知识产权局
(12) 发明专利
(10) 授权公告号 CN 115784196 B
(45) 授权公告日 2024.02.13
(21) 申请号 202211501846.4 (51) Int.Cl.
C01B 32/05(2017.01)
(22) 申请日 2022.11.28
H01M 4/133(2010.01)
(65) 同一申请的已公布的文献号
H01M 4/587(2010.01)
H01M 10/054(2010.01)
(43) 申请公布日 2023.03.14
(56) 对比文件
(73) 专利权人 湖南宸宇富基新能源科技有限公司 CN 103050663 A,2013.04.17
公司 CN 103066243 A,2013.04.24
地址 415137 湖南省常德市常德国家高新技术产业开发区西洞庭生物科技园 农
垦大道 CN 108242546 A,2018.07.03
CN 112645305 A,2021.04.13
US 2014335428 A1,2014.11.13
US 2018287207 A1,2018.10.04
(72) 发明人...
